<a href="https://colab.research.google.com/github/rcrdramalho/graduation_prediction/blob/main/ProjFinal_Cocada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Funções

##Simulação dos dados

In [ ]:
import numpy as np
def geraDados(periodo,media_disc_periodo,total_disc,n_alunos):
    """Usa uma distribuição normal baseada na média de disciplinas por período para simulados
    os dados que serão usados na aplicação. Simula n_alunos no período com quantas matérias cada
    aluno concluiu e qual foi o período em que se formou"""

    media = np.ceil((media_disc_periodo * periodo) * 0.85)                            #media da distribuição normal
    desvio_padrao = np.floor(periodo * 1.5)                                           #desvio padrão da distribuição

    lista_materias = np.random.normal(loc=media, scale=desvio_padrao, size=n_alunos)
    lista_materias = np.round(lista_materias).astype(int)                             #array gerado a partir dos parametros acima usando distribuição normal e retornando inteiros

    lista_materias = np.clip(lista_materias, 0, total_disc)                           #manter valores entre 0 e total de disciplinas para finalizar o curso

    lista_formatura = ( (total_disc - lista_materias) / 5 )                           #lista de quando se formou cada aluno simulado
    lista_formatura = (lista_formatura + periodo).astype(int)

    return(lista_materias,lista_formatura)

def geraMatriz(numero_periodos,media_disc_periodo,total_disc,n_alunos):
    """Simula vários períodos com n_alunos cada e seu período de formação"""

    M = np.zeros((numero_periodos-1,2,n_alunos),int)

    for i in range(1,numero_periodos):                                               #Para cada período gera uma lista com quantas matérias cada aluno fez e quando se formou
        array_periodo,array_formatura = geraDados(i,media_disc_periodo,total_disc,n_alunos)
        M[i-1][0] = array_periodo
        M[i-1][1] = array_formatura

    return(M)

In [ ]:
def alunos_formados(periodo,media_disc_periodo,total_disc,n_alunos):
    """Gera uma matriz com n_alunos em cada um dos periodos com o periodo do alunos
    quantas disciplinas ele possuia nesse periodo e se ele se formou antes ou durante
    o período especificado"""

    M = geraMatriz(periodo,media_disc_periodo,total_disc,n_alunos)
    periodo_materias = np.zeros(((periodo-1)*n_alunos,2),int)
    classificacao = np.zeros((periodo-1)*n_alunos,int)
    k=0
    l=1

    for i in M:                                                                    #Para cada aluno na matriz gerada, separa as materias a cada periodo e classifica se o aluno se formou ou não no período desejado
        m=0
        for j in i[1]:
            periodo_materias[k][0] = l
            periodo_materias[k][1] = i[0][m]
            if j <= periodo: classificacao[k] = 1
            else: classificacao[k] = 0
            k += 1
            m += 1
        l += 1

    return(periodo_materias,classificacao)

##Regressão Logística

In [ ]:
def funcao_logistica(z):
    """Calcula a probabilidade baseada na função de regressão logística"""
    return 1.0/(1 + np.exp(-z))

In [ ]:
def gradiente(X, y, y_hat):
    """Calcula as derivadas parciais para retornar o vetor gradiente"""
    m = X.shape[0]

    dw = (1/m)*np.dot(X.T, (y_hat - y))
    db = (1/m)*np.sum((y_hat - y))

    return dw, db

In [ ]:
def normalizar(X):
    """Normaliza as colunas de X usando a padronização (z-score)."""
    m, n = X.shape

    for i in range(n):
        X = (X - X.mean(axis=0))/X.std(axis=0)

    return X

In [ ]:
def treinamento(X, y, n, passos, alpha):
    """
    Função para treinamento de um modelo de regressão logística usando gradiente descendente.

    Parâmetros:
    - X: matriz de features (m amostras x n características)
    - y: vetor de rótulos (m amostras,)
    - n: tamanho do lote (número de amostras em cada iteração do gradiente descendente estocástico)
    - passos: número de iterações de treinamento
    - alpha: taxa de aprendizado

    Retorna:
    - w: vetor de pesos
    - b: termo de viés
    """

    m, n = X.shape

    w = np.zeros((n, 1))                                      #Pesos para a função logística inicializados como 0
    b = 0                                                     #Viés inicializado como 0

    y = y.reshape(m, 1)                                       #Garante que y seja um vetor coluna
    x = normalizar(X)

    for passo in range(passos):                               #Loop de treinamento

        for i in range((m - 1) // n + 1):                     #Usa o gradiente descendente estocástico para achar os melhores coeficientes w e b.

            inicial = i * n                                   #Índices do lote atual
            final = inicial + n

            xb = X[inicial:final]                             #Pega o lote atual(para cálculo do gradiente descendente)
            yb = y[inicial:final]

            y_hat = funcao_logistica(np.dot(xb, w) + b)       #Usa os coenficiente atuais para atualizar o valor da função logística

            dw, db = gradiente(xb, yb, y_hat)                 #Calcula os gradientes

            w -= alpha * dw                                   #Atualiza os coeficientes da função logística usando o gradiente descendente
            b -= alpha * db

    return w, b


In [ ]:
def chance_formatura(periodo_atual,n_disc,periodo_formacao,media_disc_periodo,total_disc,n_alunos):
  """Usa o treinamento para achar a melhor função logística para n_alunos em até um certo período

      Parâmetros:
    - periodo_atual: Período em que o aluno se encontra
    - n_disc: Número de disciplinas que foram concluídas até então
    - periodo_formacao: Período o qual o aluno deseja se formar
    - media_disc_periodo: Média de disciplinas do curso por período
    - total_disc: Total de disciplinas no curso
    - n_alunos: Número de alunos a serem simulados

    Retorna: a chance do aluno se formar no período desejado"""

  X, y = alunos_formados(periodo_formacao,media_disc_periodo,total_disc,n_alunos) #Gera os dados e a classificação

  w, b= treinamento(X, y, n=100, passos=100, alpha=0.01)                          #Acha os melhores coeficientes para a função logística

  return 1/(1+np.exp(-(w[0]*periodo_atual+w[1]*n_disc+b)))[0]

##Implementação

In [ ]:
#exemplo
chance_formatura(periodo_atual=3, n_disc= 15, periodo_formacao= 9, media_disc_periodo= 4, total_disc=48, n_alunos=300)
#Chance de um aluno que finalizou o 3° período com 13 matérias se formar até o 9° período (pode não estar condizente com a realidade pela forma com que os dados são simulados).

0.9756506214163514

In [ ]:
def lista_chances_formatura(periodo_atual,n_disc,max_formacao,media_disc_periodo,total_disc,n_alunos):
    """Dado um aluno em um periodo e tendo concluido um número de disciplinas, calcula a chance do aluno
    ter se formado para cada periodo a partir do oitavo até um número máximo de períodos"""
    chances = np.zeros((max_formacao-7,2))

    for i in range(8,max_formacao+1):
        chances[i-8][0] = round(chance_formatura(periodo_atual,n_disc,i, media_disc_periodo,total_disc,n_alunos),3)*100
        chances[i-8][1] = round(i)

    return chances

In [ ]:
lista_chances = lista_chances_formatura(periodo_atual=2, n_disc= 6, max_formacao= 16, media_disc_periodo= 5, total_disc=48, n_alunos=100)

In [ ]:
def printa_chances(lista_chances):
    """Função para imprimir as chances de formatura"""
    for i in lista_chances:
        print(round(i[0],1),"% de chance de se formar no ",int(i[1]),"° período.", sep='')

In [ ]:
#exemplo
printa_chances(lista_chances)
#Chance de um aluno que finalizou o segundo período com 6 matérias concluídas se formar em cada um dos períodos a partir do oitavo

0.0% de chance de se formar no 8° período.
0.5% de chance de se formar no 9° período.
56.9% de chance de se formar no 10° período.
92.7% de chance de se formar no 11° período.
97.5% de chance de se formar no 12° período.
98.1% de chance de se formar no 13° período.
98.5% de chance de se formar no 14° período.
99.0% de chance de se formar no 15° período.
99.3% de chance de se formar no 16° período.
